# 02.05 · Excel del Input CCI

Se genera `data/processed/2. Input CCI.xlsx`, con las hojas `IDOs-Soberanos` y `Listas`, a partir
de la tabla final de la cadena: `data/interim/2. IDOs - Input CCI/02.04_concepto_priorizacion/idos_concepto.csv`.
Esa tabla ya reúne, en una fila por IDO, la ficha del `02.01`, los textos del `02.02`, la
evidencia del `02.03` y la priorización del `02.04`.

La estructura, las listas y el formato del Excel se definen aquí. El notebook no lee Word ni
usa la API. El Excel es un producto generado: cada ejecución lo reemplaza.

## 1. LOAD LIBRARIES

Se usan pandas para la tabla y openpyxl para escribir las hojas, las listas desplegables y el
formato. Las rutas se toman de `caf_input_cci_preparatoria/config.py`.

In [ ]:
from datetime import date
from pathlib import Path
import re
import sys
import unicodedata

from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Se lee `idos_concepto.csv` como texto. Solo están los IDOs que pasaron por toda la cadena; si
falta alguno, hay que correr los notebooks anteriores.

In [ ]:
archivo_entrada = (
    config.INTERIM / "2. IDOs - Input CCI" / "02.04_concepto_priorizacion" / "idos_concepto.csv"
)
archivo_salida = config.PROCESSED / "2. Input CCI.xlsx"

idos = pd.read_csv(archivo_entrada, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"{len(idos)} IDOs con la cadena completa (02.01 a 02.04).")

## 3. PROCESS DATA

### Estructura del Input CCI

Se definen las 27 columnas y las siete listas que usan sus desplegables. `ID operación` y los
campos manuales quedan vacíos.

In [ ]:
columnas_input = [
    "ID operación",
    "Fecha IDO",
    "País",
    "Nombre operación",
    "Riesgo",
    "Producto",
    "Producto agrupado",
    "Monto CAF (USD MM)",
    "Capital / Aporte local (USD MM)",
    "Monto total (USD MM)",
    "Financiamiento verde",
    "Agenda",
    "Gran sector",
    "Objetivo",
    "Pertinencia + antecedentes (CTs)",
    "Intervención",
    "Evidencia",
    "Factores para maximizar el impacto (lecciones aprendidas)",
    "Decisión priorización",
    "Concepto de priorización para evaluación",
    "Enlace IDO",
    "Canal de llegada",
    "PPT / insumo evidencia enviado",
    "Llegó al CCI",
    "Año llegada CCI",
    "Revisar",
    "Archivo fuente",
]

In [ ]:
listas = {
    "Riesgo": ["Soberano", "Cuasi soberano", "No soberano"],
    "Producto agrupado": [
        "PPI",
        "PBL",
        "PpR",
        "SWAp",
        "Línea de crédito",
        "Avales y garantías",
        "Préstamo corporativo",
        "Préstamo a largo plazo",
        "Préstamo A/B",
        "Inversión patrimonial",
        "Financiamiento estructurado",
        "Liquidez",
        "Otro",
    ],
    "Financiamiento verde": ["Elegible", "Potencialmente elegible", "No elegible", "NA"],
    "Sí/No": ["Sí", "No"],
    "Sí/No/NA": ["Sí", "No", "NA"],
    "Canal": ["CCI", "Correo"],
    "Gran sector": [
        "ACCIÓN CLIMÁTICA",
        "AGRICULTURA",
        "AGUA Y SANEAMIENTO",
        "CONECTIVIDAD Y DIGITALIZACIÓN",
        "DESARROLLO PRODUCTIVO",
        "DESARROLLO URBANO",
        "DEUDA Y/O LIQUIDEZ",
        "EDUCACIÓN",
        "EDUCACIÓN SUPERIOR",
        "ELECTRIFICACIÓN",
        "GAS NATURAL",
        "GESTIÓN RIESGOS Y ATENCIÓN A DESASTRES",
        "GESTIÓN DE RESIDUOS URBANOS",
        "GOBIERNOS SUBNACIONALES",
        "INCLUSIÓN SOCIAL",
        "MOVILIDAD HUMANA",
        "PLAN DE DESARROLLO",
        "PRIMERA INFANCIA",
        "SEGURIDAD",
        "SISTEMAS PENITENCIARIOS",
        "TELEMEDICINA",
        "TRANSPORTE",
        "VIALIDAD",
        "VIVIENDA",
        "OTRO",
    ],
}

### Normalización

Se quitan las comillas exteriores del nombre de la operación. Para el riesgo se ignoran
mayúsculas, acentos y separadores al comparar con la lista.

Los montos llegan del `02.01` en millones de USD y se trasladan sin reconvertirlos. `Agenda`
copia las agendas de la ficha, con sus saltos de línea; si no hay códigos como `A3`, `B1a` o
`A.3`, se pide completarla.

In [ ]:
CODIGO_AGENDA = r"\b[ABC]\.?\d+[a-z]?\b"


def normalizar(valor):
    texto = unicodedata.normalize("NFKD", str(valor or ""))
    texto = "".join(c for c in texto if not unicodedata.combining(c)).casefold()
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()


def nombre_operacion(valor):
    texto = str(valor or "").strip()
    pares = [("“", "”"), ("«", "»"), ('"', '"'), ("'", "'")]
    while len(texto) > 1:
        par = next((p for p in pares if texto.startswith(p[0]) and texto.endswith(p[1])), None)
        if par is None:
            break
        texto = texto[1:-1].strip()
    return texto


riesgos = {normalizar(valor): valor for valor in listas["Riesgo"]}
riesgos["cuasisoberano"] = "Cuasi soberano"

### Filas del Input CCI y columna `Revisar`

`armar_fila` traslada cada IDO a las 27 columnas. `Financiamiento verde` prioriza la casilla
marcada en el Word; si el texto sugiere otra cosa, o si solo hay sugerencia, se pide validar.

`Revisar` reúne, separadas por punto y coma:

- incidencias de los datos: fecha, riesgo, montos, producto, financiamiento verde y agenda;
- alertas de los notebooks anteriores: gran sector dudoso (`02.02`), citas de evidencia no
  localizadas (`02.03`) y controles del concepto (`02.04`);
- el estado del concepto: `Decisión priorización` y `Concepto de priorización para evaluación`
  llevan lo aprobado en el `02.00` (`decision_final`, `concepto_final`). Si está `Pendiente`,
  se escribe el borrador de Azure y se avisa; si Azure redactó otro texto después de aprobarlo,
  se conserva lo aprobado y se avisa;
- la validación de los textos redactados con Azure;
- los campos que se completan a mano.

In [ ]:
campos_textos = {
    "gran_sector": "Gran sector",
    "objetivo": "Objetivo",
    "pertinencia_antecedentes": "Pertinencia + antecedentes (CTs)",
    "intervencion": "Intervención",
}
campos_evidencia = {
    "evidencia": "Evidencia",
    "factores_impacto": "Factores para maximizar el impacto (lecciones aprendidas)",
}
campos_concepto = {
    "decision_final": "Decisión priorización",
    "concepto_final": "Concepto de priorización para evaluación",
}
campos_azure = {"decision_final": "decision_sugerida", "concepto_final": "concepto"}
avisos_estado = {
    "Pendiente": "Concepto sin aprobar en 02.00: se muestra el borrador de Azure",
    "Previo": "Concepto y decisión sugeridos por Azure: validar",
}


def armar_fila(ficha):
    fila = dict.fromkeys(columnas_input)
    avisos = []
    try:
        fecha = date.fromisoformat(ficha["fecha_registro"])
    except ValueError:
        fecha = None
        avisos.append(f"Fecha IDO inválida: {ficha['fecha_registro'] or 'sin fecha'}")
    riesgo = riesgos.get(normalizar(ficha["riesgo"]))
    if riesgo is None:
        avisos.append(f"Revisar Riesgo: {ficha['riesgo'] or 'sin dato'}")
    monto = float(ficha["monto_caf"]) if ficha["monto_caf"] else None
    monto_total = float(ficha["monto_total"]) if ficha["monto_total"] else None
    if monto is None:
        avisos.append("Monto CAF vacío: revisar el aviso de montos en 02.01")
    if monto_total is None:
        avisos.append("Monto total vacío: revisar el aviso de montos en 02.01")
    if ficha["producto_agrupado"] == "Otro":
        avisos.append("Revisar Producto agrupado: modalidad sin correspondencia")
    marcado = ficha["fin_verde_marcado"]
    sugerido = ficha["fin_verde_sugerido"]
    verde = marcado or sugerido or None
    if marcado:
        if sugerido and marcado != sugerido:
            avisos.append(
                f"Financiamiento verde: casilla {marcado} pero el texto sugiere {sugerido}"
            )
    elif sugerido:
        avisos.append("Financiamiento verde sugerido desde el texto: validar")
    else:
        avisos.append("Completar Financiamiento verde")
    if not re.search(CODIGO_AGENDA, ficha["agendas"], flags=re.IGNORECASE):
        avisos.append("Completar Agenda (sin agendas en la ficha del IDO)")
    fila.update(
        {
            "Fecha IDO": fecha,
            "País": ficha["pais"].strip(),
            "Nombre operación": nombre_operacion(ficha["operacion"]),
            "Riesgo": riesgo,
            "Producto": ficha["modalidad"].strip(),
            "Producto agrupado": ficha["producto_agrupado"],
            "Monto CAF (USD MM)": monto,
            "Capital / Aporte local (USD MM)": (
                float(ficha["a_local_prestamos"]) if ficha["a_local_prestamos"] else None
            ),
            "Monto total (USD MM)": monto_total,
            "Financiamiento verde": verde,
            "Agenda": ficha["agendas"],
            "Canal de llegada": "Correo",
            "Archivo fuente": ficha["archivo"],
        }
    )

    for campo, columna in campos_textos.items():
        fila[columna] = ficha[campo].strip() or None
    if any(fila[columna] for columna in campos_textos.values()):
        avisos.append("Textos redactados por LLM: validar")
    sin_texto = [columna for columna in campos_textos.values() if not fila[columna]]
    if sin_texto:
        avisos.append("Completar textos de 02.02: " + ", ".join(sin_texto))
    if ficha["sector_dudoso"] == "True":
        avisos.append("Gran sector dudoso: validar")
    if fila["Gran sector"] and fila["Gran sector"] not in listas["Gran sector"]:
        avisos.append("Gran sector fuera de la lista: revisar clasificación")

    for campo, columna in campos_evidencia.items():
        fila[columna] = ficha[campo] if ficha[campo].strip() else None
    if ficha["citas_no_localizadas"]:
        citas = ficha["citas_no_localizadas"].replace("; ", ", ")  # "; " separa los avisos
        avisos.append(f"Citas de evidencia no localizadas: {citas}")

    estado = ficha["estado_concepto"]
    for campo, columna in campos_concepto.items():
        texto = ficha[campos_azure[campo]] if estado == "Pendiente" else ficha[campo]
        fila[columna] = texto if texto.strip() else None
    priorizacion_completa = all(fila[columna] for columna in campos_concepto.values())
    if priorizacion_completa and estado in avisos_estado:
        avisos.append(avisos_estado[estado])
    if ficha["borrador_azure_nuevo"] == "True":
        avisos.append("Azure redactó un concepto nuevo: se conserva el aprobado")
    if ficha["controles"] and estado != "Editado":
        avisos.append(f"Revisar concepto: {ficha['controles']}")

    for columna in ("País", "Nombre operación", "Producto"):
        if not fila[columna]:
            avisos.append(f"Completar {columna}")
    pendientes_evidencia = []
    if not fila["Evidencia"]:
        pendientes_evidencia.append("Evidencia")
    if not fila["Factores para maximizar el impacto (lecciones aprendidas)"]:
        pendientes_evidencia.append("Factores")
    pendientes_concepto = [] if priorizacion_completa else ["Decisión y Concepto de priorización"]
    avisos += [
        "ID operación pendiente de 02.01",
        (
            "Completar "
            + ", ".join(pendientes_evidencia + pendientes_concepto + ["Enlace IDO"])
            + ", PPT / insumo evidencia enviado, Llegó al CCI y Año llegada CCI"
        ),
    ]
    fila["Revisar"] = "; ".join(avisos)
    return fila

### Revisión antes de guardar

Las filas se ordenan por `Fecha IDO`, de la más antigua a la más reciente; con la misma fecha,
por `Archivo fuente`. Los IDOs sin fecha válida van al final. Se muestran las 27 columnas, con
filas numeradas desde 1. Los textos largos se abrevian solo en esta vista; el Excel conserva su
contenido completo.

In [ ]:
tabla = pd.DataFrame(
    [armar_fila(ficha) for ficha in idos.to_dict("records")], columns=columnas_input
).sort_values(["Fecha IDO", "Archivo fuente"], na_position="last")
tabla.index = pd.RangeIndex(1, len(tabla) + 1)
with pd.option_context(
    "display.max_columns", None, "display.max_rows", None, "display.max_colwidth", 80
):
    display(tabla)

pendientes = tabla.loc[tabla["Revisar"].ne(""), ["Archivo fuente", "Revisar"]]
print(
    f"ATENCIÓN: {len(pendientes)} de {len(tabla)} IDOs "
    "tienen campos pendientes o requieren revisión."
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(pendientes)

## 4. SAVE DATA

Se crean las hojas, los nombres de rango de `Listas` y las ocho validaciones desplegables. Se
inmoviliza `D2` para mantener visibles la fila 1 y las columnas A–C. Las fechas son fechas de
Excel y los montos son números en millones de USD. Las celdas tienen ajuste de texto, incluidos
los saltos de línea de `Agenda`. Solo se reemplaza `data/processed/2. Input CCI.xlsx`.

In [ ]:
libro = Workbook()
hoja = libro.active
hoja.title = "IDOs-Soberanos"
hoja_listas = libro.create_sheet("Listas")
hoja.append(columnas_input)
for valores in tabla.itertuples(index=False, name=None):
    hoja.append([None if pd.isna(valor) or valor == "" else valor for valor in valores])

rangos_listas = {}
for numero, (nombre, opciones) in enumerate(listas.items(), start=1):
    hoja_listas.cell(1, numero, nombre)
    for fila, opcion in enumerate(opciones, start=2):
        hoja_listas.cell(fila, numero, opcion)
    letra = get_column_letter(numero)
    nombre_rango = "lista_" + normalizar(nombre).replace(" ", "_")
    referencia = f"'Listas'!${letra}$2:${letra}${len(opciones) + 1}"
    libro.defined_names.add(DefinedName(nombre_rango, attr_text=referencia))
    rangos_listas[nombre] = nombre_rango

In [ ]:
desplegables = {
    "Riesgo": "Riesgo",
    "Producto agrupado": "Producto agrupado",
    "Financiamiento verde": "Financiamiento verde",
    "Gran sector": "Gran sector",
    "Decisión priorización": "Sí/No",
    "PPT / insumo evidencia enviado": "Sí/No/NA",
    "Llegó al CCI": "Sí/No",
    "Canal de llegada": "Canal",
}
for columna, nombre_lista in desplegables.items():
    validacion = DataValidation(
        type="list",
        formula1=f"={rangos_listas[nombre_lista]}",
        allow_blank=True,
        showDropDown=False,
        showErrorMessage=True,
        errorStyle="stop",
        errorTitle="Valor fuera de la lista",
        error="Selecciona un valor de la lista desplegable.",
    )
    letra = get_column_letter(columnas_input.index(columna) + 1)
    validacion.add(f"{letra}2:{letra}{max(2, hoja.max_row)}")
    hoja.add_data_validation(validacion)

In [ ]:
relleno_encabezado = PatternFill("solid", fgColor="1F3864")
fuente_encabezado = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
for pagina in (hoja, hoja_listas):
    for celda in pagina[1]:
        celda.fill = relleno_encabezado
        celda.font = fuente_encabezado
        celda.alignment = Alignment(wrap_text=True, vertical="center")
    for fila in pagina.iter_rows(min_row=2):
        for celda in fila:
            celda.alignment = Alignment(wrap_text=True, vertical="top")
            if isinstance(celda.value, str):
                celda.data_type = "s"

anchos = {
    "ID operación": 18,
    "Fecha IDO": 13,
    "Nombre operación": 45,
    "Objetivo": 45,
    "Pertinencia + antecedentes (CTs)": 50,
    "Intervención": 45,
    "Evidencia": 55,
    "Factores para maximizar el impacto (lecciones aprendidas)": 55,
    "Decisión priorización": 5,
    "Concepto de priorización para evaluación": 55,
    "Revisar": 35,
    "Archivo fuente": 36,
}
for numero, nombre in enumerate(columnas_input, start=1):
    hoja.column_dimensions[get_column_letter(numero)].width = anchos.get(nombre, 18)
for numero, nombre in enumerate(listas, start=1):
    hoja_listas.column_dimensions[get_column_letter(numero)].width = (
        50 if nombre == "Gran sector" else 30
    )

hoja.row_dimensions[1].height = 90
hoja_listas.row_dimensions[1].height = 32
for numero in range(2, hoja.max_row + 1):
    hoja.row_dimensions[numero].height = 105
    hoja.cell(numero, 2).number_format = "yyyy-mm-dd"
    for columna in (8, 9, 10):
        hoja.cell(numero, columna).number_format = "#,##0.0"
hoja.auto_filter.ref = f"A1:{get_column_letter(len(columnas_input))}{hoja.max_row}"
hoja.freeze_panes = "D2"
hoja_listas.freeze_panes = "A2"

In [ ]:
archivo_salida.parent.mkdir(parents=True, exist_ok=True)
libro.save(archivo_salida)
libro.close()
print(f"Guardado: {archivo_salida.relative_to(ROOT)} ({len(tabla)} IDOs, 27 columnas)")